# Run the AI Humanizer API in Google Colab

This notebook demonstrates the complete prototype in one temporary Colab runtime:

1. clone the Git repository
2. install the API and inference dependencies
3. run the mocked pytest suite
4. load the real Qwen model **once**
5. exercise the real FastAPI app with `TestClient`
6. start Uvicorn in the same process, sharing the loaded model
7. optionally create an authenticated, temporary ngrok tunnel

## Security first

- The model API is not a permanent public service.
- The notebook binds Uvicorn to `127.0.0.1` by default.
- If you create a tunnel, `POST /humanize` requires an `X-API-Key`.
- Do not share the tunnel URL or API key.
- Stop the tunnel and server when finished.
- Colab deletes packages, model weights, the server, and the URL when the runtime stops.

## Before running

- Use **Runtime > Change runtime type > T4 GPU** when available.
- Use a **fresh runtime**. If you already ran the experiments notebook, restart first
  so two copies of the model do not compete for memory.
- Edit `REPO_URL` in the clone cell.
- Optional Colab Secrets:
  - `HF_TOKEN` — only needed for private/gated models
  - `HUMANIZER_API_KEY` — API protection key
  - `NGROK_AUTHTOKEN` — only needed for the optional public tunnel


## 1. Check the runtime

We record the Python version, GPU, RAM, and disk space before downloading anything.


In [ ]:
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Colab:", Path("/content").exists())

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU detected. CPU mode will be much slower.")

try:
    import psutil

    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
except Exception:
    pass

root = Path("/content") if Path("/content").exists() else Path.cwd()
print(f"Free disk: {shutil.disk_usage(root).free / 1e9:.1f} GB")


## 2. Configure caches

Model weights stay in `/content/hf_cache`, outside the repository. That prevents
accidental commits and keeps `git status` clean.


In [ ]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

RESULTS_DIR = Path("/content/humanizer_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])


## 3. Clone or update the repository

Edit `REPO_URL`. Use a public HTTPS repository for the simplest setup. Do not put
a GitHub token in a notebook cell.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Prakash617/humanizer.git"
REPO_DIR = Path("/content/humanizer")
BRANCH = "main"

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print(
    "Commit:",
    subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
        text=True,
    ).strip(),
)


## 4. Install dependencies

`requirements-colab.txt` supplies Transformers, Accelerate, Jinja2, and textstat.
`requirements.txt` supplies FastAPI, Uvicorn, Pydantic, pytest, and httpx2.

Colab provides PyTorch. The notebook verifies it instead of replacing it with a
random CUDA wheel.


In [ ]:
%pip install -q -r /content/humanizer/requirements-colab.txt -r /content/humanizer/requirements.txt

import importlib.util
import shutil
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    command = [sys.executable, "-m", "pip", "install", "-q", "torch>=2.6,<3"]
    if shutil.which("nvidia-smi") is None:
        command.extend(["--index-url", "https://download.pytorch.org/whl/cpu"])
    subprocess.check_call(command)

import accelerate
import fastapi
import torch
import transformers
import uvicorn

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("fastapi:", fastapi.__version__)
print("uvicorn:", uvicorn.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("BF16:", torch.cuda.is_bf16_supported())


## 5. Run the mocked test suite

These tests never download a model. They verify routing, validation, status codes,
error handling, the response schema, and optional API-key protection.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd="/content/humanizer",
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()


## 6. Load the model once

The API will reuse this exact service object. It will **not** load a second model
when Uvicorn starts.

The API key is read from Colab Secrets when available. Otherwise a random key is
generated for this temporary session. `POST /humanize` requires this key because
`API_KEY` is set before the application is imported.


In [ ]:
import os
import secrets
import sys

try:
    from google.colab import userdata

    api_key = userdata.get("HUMANIZER_API_KEY")
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    api_key = os.environ.get("HUMANIZER_API_KEY")
    hf_token = os.environ.get("HF_TOKEN")

API_KEY = api_key or secrets.token_urlsafe(24)
os.environ["API_KEY"] = API_KEY
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

if "/content/humanizer" not in sys.path:
    sys.path.insert(0, "/content/humanizer")

from app.humanizer import ModelHumanizer

service = ModelHumanizer()
service.load()

print("Loaded model:", service.loaded_model_id)
print("API key (keep private):", API_KEY[:4] + "..." + API_KEY[-4:])
print("Model device:", next(service.model.parameters()).device)
print("Model dtype:", next(service.model.parameters()).dtype)


## 7. Exercise the real API with TestClient

`TestClient` sends requests directly to FastAPI without a network socket. The app
lifespan calls `service.load()` again, but it is idempotent, so the already-loaded
model is reused.

The request without a key should return `401`. The request with the key should
return `200`.


In [ ]:
import json

from fastapi.testclient import TestClient
from app.main import create_app

api_app = create_app(service)
headers = {"X-API-Key": API_KEY}

with TestClient(api_app) as client:
    root = client.get("/")
    health = client.get("/health")
    unauthorized = client.post(
        "/humanize",
        json={
            "text": "Technology plays an important role in modern education.",
            "style": "natural",
        },
    )
    response = client.post(
        "/humanize",
        json={
            "text": "Technology plays an important role in modern education.",
            "style": "natural",
        },
        headers=headers,
    )

print("GET /:", root.status_code)
print(json.dumps(root.json(), indent=2))
print("\nGET /health:", health.status_code)
print(json.dumps(health.json(), indent=2))
print("\nPOST /humanize without key:", unauthorized.status_code)
print("\nPOST /humanize with key:", response.status_code)
print(json.dumps(response.json(), indent=2))


## 8. Start Uvicorn and call the local HTTP server

Uvicorn runs in a background thread inside the same Colab process. Because it uses
the existing `api_app`, it shares the already-loaded model and does not allocate a
second copy.

Requests use `127.0.0.1`, so this is local to the Colab runtime. Nothing is public
yet.


In [ ]:
import json
import threading
import time
import urllib.error
import urllib.request

import uvicorn

config = uvicorn.Config(
    api_app,
    host="127.0.0.1",
    port=8000,
    log_level="warning",
)
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

for _ in range(60):
    if server.started:
        break
    time.sleep(0.5)
else:
    raise RuntimeError("Uvicorn did not start within 30 seconds.")

print("Uvicorn started on http://127.0.0.1:8000")

request = urllib.request.Request(
    "http://127.0.0.1:8000/humanize",
    data=json.dumps(
        {
            "text": "In 2024, the team released version 2.0 of the library.",
            "style": "professional",
        }
    ).encode("utf-8"),
    headers={
        "Content-Type": "application/json",
        "X-API-Key": API_KEY,
    },
    method="POST",
)
with urllib.request.urlopen(request, timeout=120) as response:
    print("HTTP status:", response.status)
    print(json.dumps(json.loads(response.read().decode("utf-8")), indent=2))


## 9. Optional: temporary authenticated tunnel

This is only for demonstrating HTTP access from outside the Colab runtime. It is
**not** a permanent deployment.

Requirements:

1. Create a free ngrok account and auth token.
2. Store it in Colab Secrets as `NGROK_AUTHTOKEN`.
3. The notebook refuses to create a tunnel without a token.

`POST /humanize` remains protected by `X-API-Key`. The tunnel URL and the API key
are sensitive and temporary. Never expose an unauthenticated inference endpoint.


In [ ]:
public_url = None
tunnel = None

try:
    from google.colab import userdata

    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = None

if not ngrok_token:
    print("NGROK_AUTHTOKEN is not configured. Skipping the public tunnel.")
else:
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pyngrok"]
    )
    from pyngrok import conf, ngrok

    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(8000, "http")
    public_url = tunnel.public_url

    print("Temporary public URL:", public_url)
    print("X-API-Key:", API_KEY)
    print("This URL stops working when the Colab runtime stops.")
    print("Do not share the URL or key.")


In [ ]:
if public_url is None:
    print("No tunnel to test.")
else:
    import json
    import urllib.request

    request = urllib.request.Request(
        public_url + "/humanize",
        data=json.dumps(
            {
                "text": "Remote work changed how many teams communicate.",
                "style": "natural",
            }
        ).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "X-API-Key": API_KEY,
            "ngrok-skip-browser-warning": "true",
        },
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        print("Public HTTP status:", response.status)
        print(json.dumps(json.loads(response.read().decode("utf-8")), indent=2))


## 10. Stop the server and tunnel

Always run this cell when finished. It stops the temporary URL and the local
server.


In [ ]:
public_url = globals().get("public_url")
tunnel = globals().get("tunnel")
server = globals().get("server")
server_thread = globals().get("server_thread")

if public_url:
    try:
        from pyngrok import ngrok

        ngrok.disconnect(public_url)
        ngrok.kill()
        print("Tunnel stopped.")
    except Exception as exc:
        print("Could not stop tunnel cleanly:", type(exc).__name__, exc)

if server is not None:
    server.should_exit = True
    if server_thread is not None:
        server_thread.join(timeout=15)
    print("Uvicorn stopped.")


## 11. Troubleshooting

### CUDA out of memory

- Use **Runtime > Restart runtime**, then run from the top.
- Do not run the experiments notebook and this notebook simultaneously.
- The model loader falls back to `Qwen/Qwen2.5-0.5B-Instruct` if the 1.5B model
  cannot load.

### No GPU

The API still works on CPU, but model loading and generation are much slower.
Use a GPU runtime for realistic testing.

### `401 Invalid or missing API key`

Include the header:

```text
X-API-Key: YOUR_KEY
```

The key is created in the model-loading cell. If you set
`HUMANIZER_API_KEY` in Colab Secrets, use that value.

### Uvicorn starts but requests fail

- Confirm the server printed `Uvicorn started`.
- Check that port 8000 is not already in use in the runtime.
- Run the stop cell, restart the runtime, and rerun from the top.

### ngrok URL opens an HTML warning page

For API requests add:

```text
ngrok-skip-browser-warning: true
```

This does not replace the API key.

### The runtime stopped

Packages, the Git clone, model weights, Uvicorn, the tunnel, and the API key are
gone. Re-run the notebook. Only Git source and downloaded/saved results survive.


## 12. Complete prototype checkpoint

- [ ] Mocked pytest suite passed
- [ ] Model loaded once
- [ ] `GET /` returned API metadata
- [ ] `GET /health` returned `200` and `model_loaded: true`
- [ ] `POST /humanize` without the API key returned `401`
- [ ] `POST /humanize` with the API key returned a rewrite
- [ ] Uvicorn served the same API over local HTTP
- [ ] Optional tunnel was tested with the API key
- [ ] Server and tunnel were stopped
- [ ] No model weights or secrets were committed to Git

The tunnel and Colab runtime are temporary. For a permanent deployment, use an
authenticated cloud service with a GPU, rate limits, monitoring, and a proper
frontend. Never deploy this prototype as an open, unauthenticated endpoint.
